In [1]:
%load_ext autoreload
# autoreload 2：改了 benchmark/ 下的文件不用重启 kernel
%autoreload 2

import os
os.chdir(os.path.expanduser("~/projects/LLM/hw2"))   # benchmark 包以 hw2/ 为根

import torch
from benchmark import attention_sweep, bench_attention
from cs336_basics.model import scaled_dot_product_attention as sdpa   # §4.1 的被测对象

torch.backends.cuda.matmul.allow_tf32 = False        # fp32 基准：不偷偷走 tensor core
torch.backends.cudnn.allow_tf32 = False

PROF = "profiles"                       # nsys 报告、显存快照（gitignored）
def assets(sec):                        # 每节的落盘目录 notes/assets/s<sec>；各节开头 OUT = assets(n)
    d = f"notes/assets/s{sec}"; os.makedirs(d, exist_ok=True); return d

print(torch.cuda.get_device_name(0), torch.__version__)

NVIDIA GeForce RTX 5090 2.11.0+cu130


## 4 GPU Kernels

PDF §4。编号按 problem 顺序（4.1 `pytorch_attention` = PDF 4.1.1，4.2 `torch_compile` = PDF 4.2，
4.3–4.5 三题在 PDF 里都归在 4.2.2 下）。

In [2]:
OUT = assets(4)

### 4.1 `pytorch_attention`

被测的是 hw1 的 `scaled_dot_product_attention(q, k, v, mask=None)`，
输入 `[batch, seq, d]` —— batch 固定 8，**去掉 head 维**（题面 (i)）。

每个 (d, seq) 格子量五个数：100 次前向 ms、反向开始前的 `memory_allocated()`、100 次反向 ms，外加前向峰值和全程峰值 `max_memory_allocated()`；
OOM 的格子记 `status`，不中断 sweep。

#### 表 4.1-1 / 4.1-2 / 4.1-3

三张表由 `assets/s4/tables_4_1.py` 生成：表 4.1-1 是 sweep 的透视，表 4.1-2 是 d=16 的三条显存线（反向前 / 前向峰值 / 全程峰值）；第三张逐步跑一遍 sdpa 的前向、
每步读 `memory_allocated()`，seq 16384 那列是 seq 8192 的 ×4 外推（它本身跑不到第 4 步）。

In [ ]:
import runpy
T = runpy.run_path(f"{OUT}/tables_4_1.py")
t1, t2 = T["pivot_tables"](df)
steps = T["forward_steps"](8192); steps["seq 16384 外推 GiB"] = steps.iloc[:, 2] * 4
display(t1, t2, steps)